# Agent Evaluation


    1. Build Your Agent - use Langgraph / GoogleADK / CrewAI | llm=Gemini/gpt/claude
    2. Create evaluation dataset - mlflow supports a specific format
    3. Define scorers/metrics - agentic ai specific = 
        A. Tool Selection Evaluation = ToolCallCorrectness; ToolCallEfficiency, exact Match
        B. Overall Response Quality = Correctness, completeness, Guidelines, RelevancetoQuery
        C. For Conversational Applications: Multi Turn Efficiency = ConversationCompleteness, ConversationToolCallEfficiency, UserFrustration, 
    4. Run Evaluations

## 1. Agent Implementation

In [1]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [4]:
client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()


agent = create_agent("google_genai:gemini-2.5-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='ab2ffc9c-3b6e-4805-8706-8060afca6df0'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}, '__gemini_function_call_thought_signatures__': {'d1fd863e-1913-4686-ba45-e54dce114024': 'Cq0CAQw51sc4wKqQrZrN0/h84oGTBEvAnIxIo8rVIOl5j+EPkbbYSW8WwBPk8TodrFkM39iUyHvxqvaVYlxk/lskss41vbfi6GmpD2g21jgI/3Tts5Bp89g+3z/mFuZ9VOjXAuOi8yi10Luj4eY3OkNdp5XiTL2occ4+cDEqQ48Hi0bDbDZQSwOJ6dCP9gk4GCJiZwqJFjMO3w78+7c4KDU/m/+JSOLyYyCiw0wZxa8LpvlrjcxFJ1qhRdj/u1nKArY+DnaEkwKc/JcmvlH3Q8htiohXmCTZVbkmZA36+1rA6evAJX0F6qWLeGHEoCUfxXqA0U4v4BaTuwS+vo7FAkd/hps6A5uFsNLtGashr1srOztUyw2f/T3mJbllvuLssMreeUzNvaonsKpyhrK6wg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1a8e-ab61-7672-98fa-17ffd613f759-0', tool_calls=[{'name': 'get_c

In [20]:
async def predict_function(task:str):
    response = await agent.ainvoke({"messages":[{"role":'user',"content":task}]})
    return response['messages'][-1].content

In [21]:
await predict_function("what is weather in London today?")

'The weather in London is broken clouds with a temperature of 276.07 Kelvin.'

## 2. Dataset for Agent Evaluation

In [22]:
eval_data = [{"inputs":{"task":"what is temperature in delhi today?"},
              "expectations":{"answer":"The weather in Delhi is haze with a temperature of 306.2 Kelvin."},
              "tags":{"topic":"weather"}},
              {"inputs":{"task":"what is temperature in Mumbai today?"},
              "expectations":{"answer":"The current temperature in Mumbai is 307.14 Kelvin."},
              "tags":{"topic":"weather"}},
              {"inputs":{"task":"what is temperature in London today?"},
              "expectations":{"answer":'The weather in London is broken clouds with a temperature of 276.07 Kelvin.'},
              "tags":{"topic":"weather"}},
              ]

## 3. Define Scorers

In [23]:
import mlflow
mlflow.set_tracking_uri("http://20.81.137.57:5000/")

mlflow.set_experiment("AgentEvaluation")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1778563090372, experiment_id='2', last_update_time=1778563090372, lifecycle_stage='active', name='AgentEvaluation', tags={}, trace_location=None, workspace='default'>

In [24]:
from mlflow.genai import scorer
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency


@scorer
def exact_match(outputs,expectations):
    return outputs==expectations['answer']

## 4. Run the Evaluations

In [26]:
results = mlflow.genai.evaluate(data=eval_data, predict_fn= predict_function,
                                scorers=[exact_match,ToolCallCorrectness(model="gemini:/gemini-2.5-flash"), ToolCallEfficiency(model="gemini:/gemini-2.5-flash")])

2026/05/12 10:54:21 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/05/12 10:54:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f80645ef480> was created in a different Context
2026/05/12 10:54:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f809c5ba9c0> was created in a different Context
2026/05/12 10:54:23 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f806427f900> was created in a different Context
2026/05/12 10:54:23 WARNING mlflow.utils.autologging_utils

Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]

2026/05/12 10:54:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f80648cbb00> was created in a different Context
2026/05/12 10:54:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8057eb1740> was created in a different Context
2026/05/12 10:54:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8057eb0f80> was created in a different Context
2026/05/12 10:54:27 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f809c52f880> was created in a different Context
2026/05/12 10:54:28 WARNING mlflow.utils.aut